# Dr with dataset aptos 

# all the imports and the paths that are used 

In [1]:
#imports 
import os , random 
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm 

# seed values

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


# paths of kaggle 

In [3]:
ROOT ="/kaggle/input/datasets/deepapaneru/dr-datasets/archive"
TRAIN_CSV=f"{ROOT}/train_1.csv"
VAL_CSV=f"{ROOT}/valid.csv"
TEST_CSV=f"{ROOT}/test.csv"


# laterlity path 

In [4]:
LAT_CSV = "/kaggle/input/datasets/deepapaneru/dr-datasets"
print(os.listdir(ROOT))

['val_images', 'train_images', 'valid.csv', 'test.csv', 'train_1.csv', 'test_images']


# grade csvs and picking them 

In [5]:
tr = pd.read_csv(TRAIN_CSV)
va = pd.read_csv(VAL_CSV)
te = pd.read_csv(TEST_CSV)

for name, d in [("train", tr), ("val", va), ("test", te)]:
    print(name, d.shape, list(d.columns))
print(tr.head(3))

    

train (2930, 2) ['id_code', 'diagnosis']
val (366, 2) ['id_code', 'diagnosis']
test (366, 2) ['id_code', 'diagnosis']
        id_code  diagnosis
0  1ae8c165fd53          2
1  1b329a127307          1
2  1b32e1d775ea          4


# columns name

In [6]:
ID_COL = next((c for c in tr.columns if "id" in c.lower()), tr.columns[0])
GRADE_COL = next((c for c in tr.columns
                  if c.lower() in ("diagnosis", "label", "level", "grade")), tr.columns[-1])
print("using:", ID_COL, "|", GRADE_COL)


using: id_code | diagnosis


In [7]:
#find every image on disk and stack train/val/test into one table
img_paths = {}
for folder in ["train_images", "val_images", "test_images"]:
    for dp, _, files in os.walk(f"{ROOT}/{folder}"):
        for f in files:
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                img_paths[os.path.splitext(f)[0]] = os.path.join(dp, f)
print("images found:", len(img_paths))

parts = []
for name, d in [("train", tr), ("val", va), ("test", te)]:
    p = d[[ID_COL, GRADE_COL]].copy()
    p.columns = ["id_code", "label"]
    # remove the extension in case the csv has ".png" in the id
    p["id_code"] = p["id_code"].astype(str).str.replace(r"\.\w+$", "", regex=True)
    p["split"] = name
    parts.append(p)

df = pd.concat(parts, ignore_index=True)
df["path"] = df["id_code"].map(img_paths)

print("missing images:", df.path.isna().sum())
print(df.groupby("split").size())
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

images found: 3662
missing images: 0
split
test      366
train    2930
val       366
dtype: int64
label     0    1    2    3    4
split                          
test    199   30   87   17   33
train  1434  300  808  154  234
val     172   40  104   22   28


In [8]:
#
LAT_CSV = "/kaggle/input/datasets/bikramchapagain/aptos20129-laterlity/aptos2019_laterality.csv"
lat = pd.read_csv(LAT_CSV)[["stem", "laterality", "laterality_conf", "label"]]
lat = lat.rename(columns={"stem": "id_code", "label": "label_lat"})

df = df.merge(lat, on="id_code", how="left")
print("rows with no laterality:", df.laterality.isna().sum())

# the grade in both files should be the same, quick sanity check
chk = df.dropna(subset=["label_lat"])
print("label mismatch:", int((chk.label != chk.label_lat).sum()), "of", len(chk))
df = df.drop(columns="label_lat")

print(df.groupby(["split", "laterality"]).size().unstack(fill_value=0))

rows with no laterality: 0
label mismatch: 0 of 3662
laterality  left  right  uncertain
split                             
test         180    159         27
train       1295   1438        197
val          181    153         32


In [9]:
#perceptual hash for every image
def get_hash(path):
    im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    im = cv2.resize(im, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    low = cv2.dct(im)[:8, :8]                  # keep only the low frequencies
    med = np.median(low.flatten()[1:])         # skip the first (dc) value
    return (low > med).flatten()               # 64 bits

H = np.array([get_hash(p) for p in tqdm(df.path)])
print(H.shape)

#how close is each image to its nearest other image?
n = len(df)
nearest = np.zeros(n, dtype=int)
for i in range(n):
    d = (H != H[i]).sum(axis=1)
    d[i] = 99
    nearest[i] = d.min()

print(pd.Series(nearest).value_counts().sort_index().head(12))

100%|██████████| 3662/3662 [08:02<00:00,  7.59it/s]


(3662, 64)
0     289
2     281
4     849
6     963
8     675
10    373
12    146
14     64
16     11
18      6
20      5
Name: count, dtype: int64


In [10]:
#perceptual hash for every image
def get_hash(path):
    im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    im = cv2.resize(im, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    low = cv2.dct(im)[:8, :8]                  #keep only the low frequencies
    med = np.median(low.flatten()[1:])         #skip the first (dc) value
    return (low > med).flatten()               #64 bits

H = np.array([get_hash(p) for p in tqdm(df.path)])
print(H.shape)

#how close is each image to its nearest other image?
n = len(df)
nearest = np.zeros(n, dtype=int)
for i in range(n):
    d = (H != H[i]).sum(axis=1)
    d[i] = 99
    nearest[i] = d.min()

print(pd.Series(nearest).value_counts().sort_index().head(12))

100%|██████████| 3662/3662 [05:56<00:00, 10.26it/s]


(3662, 64)
0     289
2     281
4     849
6     963
8     675
10    373
12    146
14     64
16     11
18      6
20      5
Name: count, dtype: int64


In [11]:
# cluster images whose hashes are very close
THRESH = 3   #hamming distance, look at the table above before trusting this

parent = list(range(n))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a

for i in range(n):
    d = (H != H[i]).sum(axis=1)
    d[i] = 99
    for j in np.where(d <= THRESH)[0]:
        parent[find(j)] = find(i)

df["group_id"] = ["g" + str(find(i)) for i in range(n)]

sizes = df.groupby("group_id").size()
print("groups:", len(sizes), "| images:", n)
print(sizes.value_counts().sort_index().head(10))

# groups that sit in more than one split = possible leakage
cross = df.groupby("group_id").split.nunique()
print("groups spanning >1 split:", int((cross > 1).sum()))

# groups where the grade is not the same
mixed = df.groupby("group_id").label.nunique()
print("groups with different grades:", int((mixed > 1).sum()))

groups: 3309 | images: 3662
1     3092
2      178
3       23
4        6
5        3
6        1
7        1
9        1
11       1
12       1
Name: count, dtype: int64
groups spanning >1 split: 85
groups with different grades: 63


In [12]:
#save everything E1 needs in one file
cols = ["id_code", "split", "path", "label", "laterality", "laterality_conf", "group_id"]
df[cols].to_csv("/kaggle/working/aptos_e1_master.csv", index=False)
print("saved", df.shape)

saved (3662, 7)


# settings and imports

In [13]:
#all the settings in one place so they are easy to change
import torch                                        # main deep learning library
import torch.nn as nn                               # layers and loss functions
from torch.utils.data import Dataset, DataLoader    # to feed images in batches
import torchvision                                  # the pretrained efficientnet is here
from torchvision import transforms as T             # augmentation tools
from sklearn.metrics import cohen_kappa_score       # used for QWK

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # use gpu if we have one
IMG = 512                                           # image size from the methodology
BATCH = 16                                          # batch size (lower it if memory runs out)
EPOCHS = 25                                         # max epochs
PATIENCE = 5                                        # stop if val QWK does not improve for 5 epochs
LR = 3e-4                                           # learning rate
WD = 1e-4                                           # weight decay
PREP = "both"                                       # "ben", "clahe" or "both" (for the ablation)
SEED = 42                                           # change to 123 and 2024 for the other runs

def set_seed(s):                                    # makes the run repeatable
    random.seed(s)                                  # python random
    np.random.seed(s)                               # numpy random
    torch.manual_seed(s)                            # torch random
    torch.cuda.manual_seed_all(s)                   # gpu random

set_seed(SEED)                                      # apply the seed now

m = pd.read_csv("/kaggle/working/aptos_e1_master.csv")   # the file from cell 7
m_train = m[m.split == "train"].reset_index(drop=True)   # training rows
m_val = m[m.split == "val"].reset_index(drop=True)       # validation rows
m_test = m[m.split == "test"].reset_index(drop=True)     # test rows
print(DEVICE, len(m_train), len(m_val), len(m_test))     # quick check

cuda 2930 366 366


# preprocessoing

In [14]:
# preprocessing, same for train, val and test
def crop_black(img):                                # removes the black border around the retina
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)    # gray copy to find the retina
    ys, xs = np.where(gray > 10)                    # pixels that are not black
    if len(xs) == 0:                                # if the image is fully black
        return img                                  # leave it as it is
    return img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]   # cut to the bounding box

def make_square(img):                               # pads the short side so the retina is not stretched
    h, w = img.shape[:2]                            # current height and width
    d = abs(h - w)                                  # how much padding we need
    a, b = d // 2, d - d // 2                       # split the padding on both sides
    if h > w:                                       # taller than wide
        return cv2.copyMakeBorder(img, 0, 0, a, b, cv2.BORDER_CONSTANT, value=0)   # pad left and right
    return cv2.copyMakeBorder(img, a, b, 0, 0, cv2.BORDER_CONSTANT, value=0)       # pad top and bottom

def preprocess(path):                               # full pipeline for one image
    img = cv2.imread(path)                          # read the file (BGR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)      # convert to RGB
    img = make_square(crop_black(img))              # crop border then make it square
    img = cv2.resize(img, (IMG, IMG))               # resize to 512 x 512
    if PREP in ("ben", "both"):                     # Ben Graham step
        sigma = IMG / 2 / 30                        # sigma = radius / 30
        blur = cv2.GaussianBlur(img, (0, 0), sigma) # blurred copy
        img = cv2.addWeighted(img, 4, blur, -4, 128)   # 4*I - 4*blur + 128
    if PREP in ("clahe", "both"):                   # CLAHE step
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)  # work on lightness only
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))   # settings from the methodology
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])    # boost local contrast on L channel
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)  # back to RGB
    return img                                      # uint8 image, 512 x 512 x 3

test_img = preprocess(m_train.path[0])              # try one image
print(test_img.shape, test_img.dtype)               # should be (512, 512, 3) uint8

(512, 512, 3) uint8


 # dataset, augumentation , and loaders

In [15]:
#augmentation (train only), dataset class and loaders
train_aug = T.Compose([                             # random changes, only for training images
    T.RandomHorizontalFlip(),                       # random left-right flip
    T.RandomVerticalFlip(),                         # random up-down flip
    T.RandomRotation(30),                           # rotate up to 30 degrees
    T.ColorJitter(brightness=0.2, contrast=0.2),    # small brightness and contrast change
])                                                  # no blur, as decided in the literature table

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)   # ImageNet mean
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)    # ImageNet std

class DRData(Dataset):                              # gives one image and its labels at a time
    def __init__(self, table, train):               # table = dataframe, train = use augmentation or not
        self.paths = table.path.tolist()            # list of image paths
        self.labels = table.label.astype(int).tolist()   # list of grades 0-4
        self.train = train                          # remember the mode

    def __len__(self):                              # number of images
        return len(self.paths)

    def __getitem__(self, i):                       # load image number i
        img = preprocess(self.paths[i])             # run the preprocessing
        x = torch.from_numpy(img).permute(2, 0, 1)  # to channel-first tensor (uint8)
        if self.train:                              # augmentation only while training
            x = train_aug(x)
        x = x.float() / 255.0                       # scale to 0-1
        x = (x - MEAN) / STD                        # normalise like ImageNet
        y = self.labels[i]                          # the grade
        levels = torch.tensor([float(y > k) for k in range(4)])   # ordinal targets: is grade > 0, > 1, > 2, > 3
        return x, levels, y                         # image, ordinal targets, plain grade

train_loader = DataLoader(DRData(m_train, True), batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)   # shuffled for training
val_loader = DataLoader(DRData(m_val, False), batch_size=BATCH, shuffle=False,
                        num_workers=2, pin_memory=True)                     # fixed order for val
test_loader = DataLoader(DRData(m_test, False), batch_size=BATCH, shuffle=False,
                         num_workers=2, pin_memory=True)                    # fixed order for test

xb, lb, yb = next(iter(train_loader))               # pull one batch to check shapes
print(xb.shape, lb.shape, yb[:8])                   # (16,3,512,512) (16,4)

torch.Size([16, 3, 512, 512]) torch.Size([16, 4]) tensor([2, 0, 0, 0, 4, 4, 0, 0])


# model and loss

In [16]:
# EfficientNet-B0 with a CORAL ordinal head
class CoralEffNet(nn.Module):
    def __init__(self):
        super().__init__()                          # start the parent class
        base = torchvision.models.efficientnet_b0(weights="IMAGENET1K_V1")   # pretrained (needs internet ON in Kaggle)
        self.features = base.features               # keep only the convolution part
        self.pool = nn.AdaptiveAvgPool2d(1)         # global average pooling
        self.drop = nn.Dropout(0.3)                 # some dropout against overfitting
        self.fc = nn.Linear(1280, 1, bias=False)    # one shared score z = w^T h
        self.bias = nn.Parameter(torch.linspace(1.5, -1.5, 4))   # 4 thresholds, start in decreasing order

    def forward(self, x):
        h = self.pool(self.features(x)).flatten(1)  # 1280-d feature per image
        z = self.fc(self.drop(h))                   # shared score, shape (batch, 1)
        return z + self.bias                        # logits for the 4 thresholds, shape (batch, 4)

counts = m_train.label.astype(int).value_counts().sort_index()      # images per grade in train
CLASS_W = torch.tensor((len(m_train) / (5 * counts)).values,
                       dtype=torch.float32).to(DEVICE)              # omega_c = N / (K * n_c)
print(CLASS_W)                                      # rare grades should get bigger weights

def coral_loss(logits, levels, grades):             # weighted CORAL loss
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits, levels, reduction="none").sum(1)    # binary loss summed over the 4 thresholds
    w = CLASS_W[grades]                             # weight of each sample by its true grade
    return (bce * w).mean()                         # average over the batch

def to_grade(logits):                               # logits -> predicted grade 0-4
    return (torch.sigmoid(logits) > 0.5).sum(1)     # count how many thresholds are passed

tensor([0.4086, 1.9533, 0.7252, 3.8052, 2.5043], device='cuda:0')


# train and evaluate helpers

In [17]:
# helper functions for one epoch and for prediction
def predict(model, loader):                         # returns probabilities and true grades
    model.eval()                                    # evaluation mode (no dropout)
    all_p, all_y = [], []                           # collectors
    with torch.no_grad():                           # no gradients needed
        for x, _, y in loader:
            x = x.to(DEVICE)                        # move images to gpu
            with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):   # mixed precision
                out = model(x)                      # logits
            all_p.append(torch.sigmoid(out.float()).cpu())   # threshold probabilities
            all_y.append(y)                         # true grades
    return torch.cat(all_p).numpy(), torch.cat(all_y).numpy()   # as numpy arrays

def qwk(p, y):                                      # quadratic weighted kappa from threshold probs
    pred = (p > 0.5).sum(1)                         # predicted grades
    return cohen_kappa_score(y, pred, weights="quadratic")   # QWK

def train_model(seed):                              # full training run for one seed
    set_seed(seed)                                  # repeatable run
    model = CoralEffNet().to(DEVICE)                # new model
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)   # optimiser
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader))  # LR goes up then down
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")       # for mixed precision
    best, bad = -1, 0                               # best val QWK and the patience counter
    save_to = f"/kaggle/working/e1_b0_seed{seed}.pt"    # where the best weights are saved
    log = []                                        # keeps the numbers of each epoch

    for ep in range(EPOCHS):
        model.train()                               # training mode
        total = 0                                   # running loss
        for x, lv, y in tqdm(train_loader, desc=f"epoch {ep+1}", leave=False):
            x, lv, y = x.to(DEVICE), lv.to(DEVICE), y.to(DEVICE)   # move to gpu
            opt.zero_grad()                         # clear old gradients
            with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
                loss = coral_loss(model(x).float(), lv, y)         # forward pass and loss
            scaler.scale(loss).backward()           # backward pass
            scaler.step(opt)                        # update weights
            scaler.update()                         # update the scaler
            sched.step()                            # move the learning rate
            total += loss.item()                    # add up the loss
        pv, yv = predict(model, val_loader)         # check on the validation set
        k = qwk(pv, yv)                             # val QWK
        log.append((ep + 1, total / len(train_loader), k))         # save the numbers
        print(f"epoch {ep+1:02d} | train loss {total/len(train_loader):.4f} | val QWK {k:.4f}")
        if k > best:                                # improved
            best, bad = k, 0                        # reset the counter
            torch.save(model.state_dict(), save_to) # keep these weights
        else:
            bad += 1                                # no improvement
            if bad >= PATIENCE:                     # out of patience
                print("early stop")
                break
    print("best val QWK:", round(best, 4))
    return save_to, log                             # path of the best weights and the log

# run training

In [18]:
 # run training
weights_path, history = train_model(SEED)           # this is the long cell
pd.DataFrame(history, columns=["epoch", "train_loss", "val_qwk"]).to_csv(
    f"/kaggle/working/e1_log_seed{SEED}.csv", index=False)    # save the log

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 141MB/s] 


epoch 01 | train loss 2.0937 | val QWK 0.6233


epoch 02 | train loss 1.5892 | val QWK 0.8511


epoch 03 | train loss 1.3241 | val QWK 0.8615


...

epoch 14 | train loss 0.9132 | val QWK 0.8963


epoch 15 | train loss 0.8745 | val QWK 0.8752


epoch 16 | train loss 0.8524 | val QWK 0.9096


epoch 17 | train loss 0.8295 | val QWK 0.9098


epoch 18 | train loss 0.8082 | val QWK 0.8808
early stop
best val QWK: 0.9202


# E1 test (APTOS test split)

In [19]:

from sklearn.metrics import recall_score, f1_score, roc_auc_score, balanced_accuracy_score

model = CoralEffNet().to(DEVICE)                    # empty model
model.load_state_dict(torch.load(weights_path))     # load the best weights
pt, yt = predict(model, test_loader)                # test predictions

pred = (pt > 0.5).sum(1)                            # predicted grades
cp = np.zeros((len(pt), 5))                         # class probabilities from the thresholds
cp[:, 0] = 1 - pt[:, 0]                             # P(y=0)
for k in range(1, 4):
    cp[:, k] = pt[:, k - 1] - pt[:, k]              # P(y=k)
cp[:, 4] = pt[:, 3]                                 # P(y=4)
cp = np.clip(cp, 1e-6, None)                        # remove tiny negatives
cp = cp / cp.sum(1, keepdims=True)                  # make rows sum to 1

print("accuracy        :", round((pred == yt).mean(), 4))
print("QWK             :", round(cohen_kappa_score(yt, pred, weights="quadratic"), 4))
print("balanced acc    :", round(balanced_accuracy_score(yt, pred), 4))
print("macro F1        :", round(f1_score(yt, pred, average="macro"), 4))
print("macro AUC (ovr) :", round(roc_auc_score(yt, cp, multi_class="ovr", labels=[0, 1, 2, 3, 4]), 4))
print("per-class sensitivity:", dict(zip(range(5), recall_score(yt, pred, average=None, labels=[0, 1, 2, 3, 4]).round(3))))

ref_true, ref_pred = yt >= 2, pred >= 2             # referable DR = grade 2 or more
sens = (ref_pred & ref_true).sum() / ref_true.sum() # referable sensitivity
spec = (~ref_pred & ~ref_true).sum() / (~ref_true).sum()   # referable specificity
print("referable DR sens/spec:", round(sens, 3), round(spec, 3))

out = m_test[["id_code", "label", "laterality", "group_id"]].copy()   # save predictions for CIs later
out["pred"] = pred
for k in range(5):
    out[f"p{k}"] = cp[:, k]
out.to_csv(f"/kaggle/working/e1_test_preds_seed{SEED}.csv", index=False)

accuracy        : 0.7705
QWK             : 0.9016
balanced acc    : 0.6087
macro F1        : 0.5863
macro AUC (ovr) : 0.9305
per-class sensitivity: {0: np.float64(0.975), 1: np.float64(0.633), 2: np.float64(0.54), 3: np.float64(0.471), 4: np.float64(0.424)}
referable DR sens/spec: 0.839 0.969
